In [ ]:
# # This Python 3 environment comes with many helpful analytics libraries installed
# # It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# # For example, here's several helpful packages to load

# import numpy as np # linear algebra
# import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# # Input data files are available in the read-only "../input/" directory
# # For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# # You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# # You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import shutil

BASE_DIR = Path("/kaggle/input/datasets/ymirsky/network-attack-dataset-kitsune")
OUT_DIR = Path("/kaggle/working/kitsune_20pct_samples")
OUT_DIR.mkdir(parents=True, exist_ok=True)

FRAC = 0.20
SEED = 42
CHUNKSIZE = 100_000

if not BASE_DIR.exists():
    raise FileNotFoundError(f"Base directory not found: {BASE_DIR}")

subfolders = [p for p in BASE_DIR.iterdir() if p.is_dir()]
if not subfolders:
    raise FileNotFoundError("No subfolders found.")

manifest = []

for folder in sorted(subfolders):
    attack_name = folder.name.replace(" ", "_")

    dataset_files = list(folder.glob("*_dataset.csv"))
    label_files = list(folder.glob("*_labels.csv"))

    if len(dataset_files) != 1 or len(label_files) != 1:
        print(f"Skipping {folder.name}: expected 1 dataset file and 1 labels file")
        continue

    dataset_path = dataset_files[0]
    labels_path = label_files[0]

    out_data = OUT_DIR / f"{attack_name}_20pct_dataset.csv"
    out_labels = OUT_DIR / f"{attack_name}_20pct_labels.csv"

    if out_data.exists():
        out_data.unlink()
    if out_labels.exists():
        out_labels.unlink()

    data_reader = pd.read_csv(
        dataset_path,
        header=None,
        chunksize=CHUNKSIZE,
        low_memory=False
    )

    # Handling for Botnet as it is the only menance with extra column of index ;-;
    if folder.name == "Mirai Botnet":
        label_reader = pd.read_csv(
            labels_path,
            header=None,
            chunksize=CHUNKSIZE,
            low_memory=False
        )
    else:
        label_reader = pd.read_csv(
            labels_path,
            header=0,
            chunksize=CHUNKSIZE,
            low_memory=False
        )

    total_rows = 0
    sampled_rows = 0

    for chunk_id, (data_chunk, label_chunk) in enumerate(zip(data_reader, label_reader)):
        if label_chunk.shape[1] == 1:
            labels_only = label_chunk.iloc[:, [0]]
        else:
            labels_only = label_chunk.iloc[:, [-1]]

        if len(data_chunk) != len(labels_only):
            raise ValueError(
                f"Row mismatch in {folder.name}, chunk {chunk_id}: "
                f"data={len(data_chunk)}, labels={len(labels_only)}"
            )

        rng = np.random.default_rng(SEED + chunk_id)
        mask = rng.random(len(data_chunk)) < FRAC

        sampled_data = data_chunk.loc[mask]
        sampled_labels = labels_only.loc[mask]

        sampled_data.to_csv(out_data, mode="a", index=False, header=False)
        sampled_labels.to_csv(out_labels, mode="a", index=False, header=False)

        total_rows += len(data_chunk)
        sampled_rows += len(sampled_data)

    manifest.append({
        "attack_folder": folder.name,
        "dataset_file": str(dataset_path),
        "labels_file": str(labels_path),
        "sampled_dataset_file": str(out_data),
        "sampled_labels_file": str(out_labels),
        "original_rows": total_rows,
        "sampled_rows": sampled_rows,
        "sample_fraction": sampled_rows / total_rows if total_rows else 0
    })

    print(f"Done: {folder.name}")
    print(f"  Original rows : {total_rows}")
    print(f"  Sampled rows  : {sampled_rows}")
    print(f"  Dataset saved : {out_data}")
    print(f"  Labels saved  : {out_labels}")
    print("-" * 60)

manifest_df = pd.DataFrame(manifest)
manifest_path = OUT_DIR / "kitsune_20pct_manifest.csv"
manifest_df.to_csv(manifest_path, index=False)

print("\nAll datasets processed.")
print(f"Manifest saved to: {manifest_path}")
print(manifest_df)

zip_path = "/kaggle/working/kitsune_20pct_samples_bundle"
shutil.make_archive(zip_path, "zip", OUT_DIR)
print(f"Zip created: {zip_path}.zip")


In [ ]:
import os

base = "/kaggle/working/project"
os.makedirs(f"{base}/core", exist_ok=True)
os.makedirs(f"{base}/evaluation", exist_ok=True)
os.makedirs(f"{base}/results", exist_ok=True)

In [14]:
%%writefile /kaggle/working/project/core/kitnet.py
from __future__ import annotations
import numpy as np


class Autoencoder:
    # Three-layer autoencoder trained online with SGD.
    # Normalisation bounds are recorded during training and frozen at inference.

    def __init__(self, n_visible: int, n_hidden: int, lr: float = 0.1):
        self.n_visible = n_visible
        self.n_hidden  = n_hidden
        self.lr        = lr

        bound = 1.0 / max(n_visible, 1)
        rng   = np.random.default_rng()
        self.W_enc = rng.uniform(-bound, bound, (n_hidden,  n_visible)).astype(np.float64)
        self.b_enc = np.zeros(n_hidden,  dtype=np.float64)
        self.W_dec = rng.uniform(-bound, bound, (n_visible, n_hidden )).astype(np.float64)
        self.b_dec = np.zeros(n_visible, dtype=np.float64)

        self._feat_min = np.full(n_visible,  np.inf, dtype=np.float64)
        self._feat_max = np.full(n_visible, -np.inf, dtype=np.float64)

    def _record_bounds(self, x: np.ndarray) -> None:
        self._feat_min = np.minimum(self._feat_min, x)
        self._feat_max = np.maximum(self._feat_max, x)

    def _normalise(self, x: np.ndarray) -> np.ndarray:
        span = self._feat_max - self._feat_min
        span[span == 0] = 1.0
        return (x - self._feat_min) / span

    @staticmethod
    def _sigmoid(z: np.ndarray) -> np.ndarray:
        z = np.clip(z, -500.0, 500.0)
        return np.where(z >= 0, 1.0 / (1.0 + np.exp(-z)), np.exp(z) / (1.0 + np.exp(z)))

    @staticmethod
    def _sigmoid_grad(a: np.ndarray) -> np.ndarray:
        return a * (1.0 - a)

    def _forward(self, x_norm: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
        hidden = self._sigmoid(self.W_enc @ x_norm + self.b_enc)
        output = self._sigmoid(self.W_dec @ hidden  + self.b_dec)
        return hidden, output

    @staticmethod
    def _rmse(original: np.ndarray, reconstructed: np.ndarray) -> float:
        return float(np.sqrt(np.mean((original - reconstructed) ** 2)))

    def train(self, x: np.ndarray) -> float:
        self._record_bounds(x)
        x_norm = self._normalise(x)
        hidden, output = self._forward(x_norm)
        error = self._rmse(x_norm, output)

        # Backpropagation through output then hidden layer
        out_delta    = (output - x_norm) * self._sigmoid_grad(output)
        hidden_delta = (self.W_dec.T @ out_delta) * self._sigmoid_grad(hidden)

        self.W_dec -= self.lr * np.outer(out_delta,    hidden)
        self.b_dec -= self.lr * out_delta
        self.W_enc -= self.lr * np.outer(hidden_delta, x_norm)
        self.b_enc -= self.lr * hidden_delta

        return error

    def execute(self, x: np.ndarray) -> float:
        # Inference only — no weight update.
        x_norm = self._normalise(x)
        _, output = self._forward(x_norm)
        return self._rmse(x_norm, output)

    # Backward-compatible aliases
    @property
    def W1(self) -> np.ndarray:
        return self.W_enc

    @property
    def W2(self) -> np.ndarray:
        return self.W_dec

    @property
    def n_v(self) -> int:
        return self.n_visible

    @property
    def n_h(self) -> int:
        return self.n_hidden


class KitNET:
    # Ensemble of small autoencoders (one per feature cluster) plus
    # an output autoencoder that aggregates their RMSE errors.

    def __init__(self, cluster_sizes: list[int], beta: float = 0.75, lr: float = 0.1):
        self.k    = len(cluster_sizes)
        self.beta = beta
        self.lr   = lr

        self.ensemble: list[Autoencoder] = [
            Autoencoder(sz, max(1, int(np.ceil(beta * sz))), lr)
            for sz in cluster_sizes
        ]
        self.output_layer = Autoencoder(self.k, max(1, int(np.ceil(beta * self.k))), lr)
        self.phi: float = 0.0

    def train(self, sub_instances: list[np.ndarray]) -> float:
        error_signal = np.zeros(self.k, dtype=np.float64)
        for i, (ae, vi) in enumerate(zip(self.ensemble, sub_instances)):
            error_signal[i] = ae.train(vi)
        score    = self.output_layer.train(error_signal)
        self.phi = max(self.phi, score)
        return score

    def execute(self, sub_instances: list[np.ndarray]) -> float:
        error_signal = np.zeros(self.k, dtype=np.float64)
        for i, (ae, vi) in enumerate(zip(self.ensemble, sub_instances)):
            error_signal[i] = ae.execute(vi)
        return self.output_layer.execute(error_signal)

    @property
    def output_ae(self) -> Autoencoder:
        return self.output_layer



Overwriting /kaggle/working/project/core/kitnet.py


In [ ]:
%%writefile /kaggle/working/project/core/feature_mapper.py
"""
core/feature_mapper.py
-----------------------
Feature Mapper (FM) component of Kitsune.

During training: incrementally maintains a partial correlation matrix C over
the n feature dimensions by accumulating running sums needed to compute
pairwise correlation distances without storing any individual instances.

After training: performs hierarchical agglomerative clustering on the nxn
correlation distance matrix D and cuts the dendrogram to yield k groups,
each of size ≤ m.  The resulting mapping f(x) = v partitions every future
feature vector into k sub-instances, one per autoencoder in the ensemble.

Paper reference: Mirsky et al., NDSS 2018, Section IV-C (Equations 8-10).
"""

from __future__ import annotations
import numpy as np
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform


class FeatureMapper:
    """
    Parameters
    ----------
    n_features : int
        Total number of input features (115 for original Kitsune).
    max_cluster_size : int
        Maximum number of features per sub-instance (m in the paper).
        Controls the ensemble size and per-autoencoder complexity.
    """

    def __init__(self, n_features: int = 115, max_cluster_size: int = 10):
        self.n = n_features
        self.m = max_cluster_size

        # Incremental partial-correlation accumulators
        # c[i]   = running sum of feature i values
        # c_rs[i]= running sum of squared residuals for feature i
        # C[i,j] = running sum of products of residuals for features i,j
        self._c    = np.zeros(self.n, dtype=np.float64)
        self._c_rs = np.zeros(self.n, dtype=np.float64)
        self._C    = np.zeros((self.n, self.n), dtype=np.float64)
        self._t    = 0  # number of instances seen

        # Populated after fit() is called
        self.mapping: list[list[int]] | None = None  # k groups of feature indices
        self.is_fitted: bool = False

    # ------------------------------------------------------------------ #
    # Training phase
    # ------------------------------------------------------------------ #

    def update(self, x: np.ndarray) -> None:
        """
        Incorporate one feature vector into the incremental correlation matrix.
        Must be called for every training instance before fit() is invoked.

        Parameters
        ----------
        x : np.ndarray, shape (n,)
        """
        self._t += 1
        # Running sums
        self._c += x
        # Current mean estimate (used only to compute residual)
        mu = self._c / self._t
        r = x - mu                        # residual vector, shape (n,)
        self._c_rs += r ** 2              # per-feature squared residuals
        # Outer product of residuals → update cross-feature products
        self._C    += np.outer(r, r)

    # ------------------------------------------------------------------ #
    # Derive distance matrix and cluster
    # ------------------------------------------------------------------ #

    def fit(self) -> None:
        """
        Compute the nXn correlation distance matrix D from the accumulated
        statistics, perform hierarchical clustering, and cut the dendrogram
        to obtain groups of size ≤ m.

        Must be called exactly once after all training instances have been
        passed through update().
        """
        # Equation (10) in the paper: D[i,j] = 1 - C[i,j] / sqrt(c_rs[i] * c_rs[j])
        D = np.ones((self.n, self.n), dtype=np.float64)

        denom_outer = np.sqrt(np.outer(self._c_rs, self._c_rs))
        # Avoid divide-by-zero for constant features
        nonzero = denom_outer > 1e-12
        D[nonzero] = 1.0 - self._C[nonzero] / denom_outer[nonzero]

        # Clip to [0, 1] to handle minor numerical issues
        D = np.clip(D, 0.0, 1.0)
        np.fill_diagonal(D, 0.0)

        # Symmetrise (should already be symmetric, but numerical safety)
        D = (D + D.T) / 2.0

        # Hierarchical agglomerative clustering with complete linkage
        # (ensures tight, well-bounded clusters)
        condensed = squareform(D, checks=False)
        Z = linkage(condensed, method="complete")

        # Iteratively increase the cut threshold until all
        # clusters have size ≤ m, following the paper's procedure.
        self.mapping = self._cut_dendrogram(Z, D)
        self.is_fitted = True

    def _cut_dendrogram(
        self, Z: np.ndarray, D: np.ndarray
    ) -> list[list[int]]:
        """
        Cut till no group exceeds size m.

        Strategy: try increasing distance thresholds from 0 to 1 in small
        steps; for any cluster that still exceeds m, recursively split it
        by halving the threshold within that sub-cluster.
        """
        # Start: every feature is its own cluster
        labels = fcluster(Z, t=self.m, criterion="maxclust")

        # Re-partition to respect the size bound m exactly
        groups: dict[int, list[int]] = {}
        for feat_idx, cluster_id in enumerate(labels):
            groups.setdefault(cluster_id, []).append(feat_idx)

        # If any group still exceeds m, split greedily
        final_groups: list[list[int]] = []
        for grp in groups.values():
            if len(grp) <= self.m:
                final_groups.append(grp)
            else:
                # Split into chunks of size m
                for i in range(0, len(grp), self.m):
                    final_groups.append(grp[i:i + self.m])

        return final_groups

    # Exec phase

    def transform(self, x: np.ndarray) -> list[np.ndarray]:
        """
        Apply the learned mapping to a feature vector.

        Returns
        -------
        v : list of np.ndarray
            k sub-instances, one per cluster/autoencoder.

        Raises
        ------
        RuntimeError if fit() has not been called.
        """
        if not self.is_fitted:
            raise RuntimeError("FeatureMapper.fit() must be called before transform().")
        return [x[idx] for idx in self.mapping]

    # Convenience properties

    @property
    def n_clusters(self) -> int:
        """Number of feature groups (k) after fitting."""
        if not self.is_fitted:
            return 0
        return len(self.mapping)

    @property
    def cluster_sizes(self) -> list[int]:
        if not self.is_fitted:
            return []
        return [len(g) for g in self.mapping]


In [ ]:
%%writefile /kaggle/working/project/evaluation/metrics.py
from __future__ import annotations

import logging
from typing import Any

import numpy as np
import pandas as pd
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)

logger = logging.getLogger(__name__)


def _tpr_at_fpr(
    fpr_arr: np.ndarray,
    tpr_arr: np.ndarray,
    thresholds: np.ndarray,
    target_fpr: float,
) -> tuple[float, float, float]:
    valid = np.where(fpr_arr <= target_fpr + 1e-12)[0]
    if len(valid) == 0:
        thr = float(thresholds[0]) if len(thresholds) else 0.0
        return 0.0, 1.0, thr

    best_idx = valid[np.argmax(tpr_arr[valid])]
    tpr = float(tpr_arr[best_idx])
    fnr = 1.0 - tpr
    thr = float(thresholds[best_idx]) if best_idx < len(thresholds) else 0.0
    return tpr, fnr, thr


def _eer_from_roc(fpr_arr: np.ndarray, tpr_arr: np.ndarray) -> float:
    fnr_arr = 1.0 - tpr_arr
    idx = int(np.argmin(np.abs(fpr_arr - fnr_arr)))
    return float((fpr_arr[idx] + fnr_arr[idx]) / 2.0)


def _best_f1_threshold(scores: np.ndarray, labels: np.ndarray) -> tuple[float, float]:
    precision_arr, recall_arr, thresholds = precision_recall_curve(labels, scores)

    if len(thresholds) == 0:
        thr = float(np.median(scores))
        preds = (scores >= thr).astype(int)
        return thr, float(f1_score(labels, preds, zero_division=0))

    f1_arr = (
        2.0 * precision_arr[:-1] * recall_arr[:-1]
        / (precision_arr[:-1] + recall_arr[:-1] + 1e-12)
    )

    best_idx = int(np.nanargmax(f1_arr))
    return float(thresholds[best_idx]), float(f1_arr[best_idx])


def build_curve_frames(scores: np.ndarray, labels: np.ndarray) -> tuple[pd.DataFrame, pd.DataFrame]:
    scores = np.asarray(scores, dtype=np.float64)
    labels = np.asarray(labels, dtype=np.int32)

    if len(np.unique(labels)) < 2:
        roc_df = pd.DataFrame({"fpr": [0.0, 1.0], "tpr": [0.0, 1.0]})
        pr_df = pd.DataFrame({"recall": [0.0, 1.0], "precision": [1.0, 0.0]})
        return roc_df, pr_df

    fpr_arr, tpr_arr, _ = roc_curve(labels, scores, pos_label=1)
    precision_arr, recall_arr, _ = precision_recall_curve(labels, scores)

    roc_df = pd.DataFrame({"fpr": fpr_arr, "tpr": tpr_arr})
    pr_df = pd.DataFrame({"recall": recall_arr, "precision": precision_arr})
    return roc_df, pr_df


def compute_metrics(
    scores: np.ndarray,
    labels: np.ndarray,
    dataset_name: str = "",
    runtime_sec: float | None = None,
    extra: dict[str, Any] | None = None,
) -> dict[str, Any]:
    scores = np.asarray(scores, dtype=np.float64)
    labels = np.asarray(labels, dtype=np.int32)

    n_total = int(len(labels))
    n_pos = int(labels.sum())
    n_neg = int((labels == 0).sum())

    if n_total == 0:
        return {"dataset": dataset_name, "error": "no_eval_rows"}

    if n_pos == 0:
        logger.warning("I found no attack rows for %s.", dataset_name)
        return {"dataset": dataset_name, "error": "no_positives"}

    if n_neg == 0:
        logger.warning("I found no benign rows for %s.", dataset_name)
        return {"dataset": dataset_name, "error": "no_negatives"}

    fpr_arr, tpr_arr, thresholds = roc_curve(labels, scores, pos_label=1)
    auc = float(roc_auc_score(labels, scores))
    auprc = float(average_precision_score(labels, scores))

    tpr0, fnr0, thr0 = _tpr_at_fpr(fpr_arr, tpr_arr, thresholds, 0.0)
    tpr001, fnr001, thr001 = _tpr_at_fpr(fpr_arr, tpr_arr, thresholds, 0.001)

    eer = _eer_from_roc(fpr_arr, tpr_arr)
    best_thr, best_f1 = _best_f1_threshold(scores, labels)

    preds = (scores >= best_thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(labels, preds, labels=[0, 1]).ravel()

    precision_opt = float(precision_score(labels, preds, zero_division=0))
    recall_opt = float(recall_score(labels, preds, zero_division=0))

    benign_scores = scores[labels == 0]
    attack_scores = scores[labels == 1]

    metrics = {
        "dataset": dataset_name,
        "n_total": n_total,
        "n_benign": n_neg,
        "n_malicious": n_pos,
        "attack_rate": round(float(labels.mean()), 6),
        "AUC": round(auc, 6),
        "AUPRC": round(auprc, 6),
        "EER": round(eer, 6),
        "TPR_at_FPR_0": round(tpr0, 6),
        "FNR_at_FPR_0": round(fnr0, 6),
        "threshold_FPR_0": round(thr0, 6),
        "TPR_at_FPR_0001": round(tpr001, 6),
        "FNR_at_FPR_0001": round(fnr001, 6),
        "threshold_FPR_0001": round(thr001, 6),
        "threshold_opt": round(best_thr, 6),
        "F1_optimal": round(best_f1, 6),
        "Precision_opt": round(precision_opt, 6),
        "Recall_opt": round(recall_opt, 6),
        "TP": int(tp),
        "FP": int(fp),
        "FN": int(fn),
        "TN": int(tn),
        "mean_score_benign": round(float(np.mean(benign_scores)), 6),
        "mean_score_attack": round(float(np.mean(attack_scores)), 6),
        "median_score_benign": round(float(np.median(benign_scores)), 6),
        "median_score_attack": round(float(np.median(attack_scores)), 6),
        "std_score_benign": round(float(np.std(benign_scores)), 6),
        "std_score_attack": round(float(np.std(attack_scores)), 6),
        "max_score": round(float(np.max(scores)), 6),
        "min_score": round(float(np.min(scores)), 6),
    }

    if runtime_sec is not None:
        metrics["runtime_sec"] = round(float(runtime_sec), 4)
        metrics["rows_per_sec"] = round(float(n_total / runtime_sec), 4) if runtime_sec > 0 else 0.0

    if extra:
        metrics.update(extra)

    return metrics


def print_metrics(metrics: dict[str, Any]) -> None:
    if "error" in metrics:
        logger.warning("Metrics unavailable for %s: %s", metrics.get("dataset", "dataset"), metrics["error"])
        return

    logger.info("─" * 60)
    logger.info("Dataset: %s", metrics["dataset"])
    logger.info(
        "Eval rows: %d | Benign: %d | Attack: %d",
        metrics["n_total"],
        metrics["n_benign"],
        metrics["n_malicious"],
    )
    logger.info(
        "AUC: %.4f | AUPRC: %.4f | EER: %.4f",
        metrics["AUC"],
        metrics["AUPRC"],
        metrics["EER"],
    )
    logger.info(
        "TPR@FPR=0: %.4f | TPR@FPR=0.001: %.4f",
        metrics["TPR_at_FPR_0"],
        metrics["TPR_at_FPR_0001"],
    )
    logger.info(
        "F1: %.4f | Precision: %.4f | Recall: %.4f",
        metrics["F1_optimal"],
        metrics["Precision_opt"],
        metrics["Recall_opt"],
    )
    logger.info(
        "TP=%d FP=%d FN=%d TN=%d",
        metrics["TP"],
        metrics["FP"],
        metrics["FN"],
        metrics["TN"],
    )
    if "runtime_sec" in metrics:
        logger.info(
            "Runtime: %.2fs | Rows/sec: %.2f",
            metrics["runtime_sec"],
            metrics["rows_per_sec"],
        )
    logger.info("─" * 60)


In [ ]:
%%writefile /kaggle/working/project/evaluation/plot_results.py
from __future__ import annotations

import argparse
import json
import logging
from pathlib import Path

import matplotlib
matplotlib.use("Agg")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

logger = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")


def _result_dirs(results_dir: Path) -> list[Path]:
    return sorted(
        [
            d for d in results_dir.iterdir()
            if d.is_dir()
            and (d / "metrics.json").exists()
            and (d / "scores.csv").exists()
        ]
    )


def load_summary_metrics(results_dir: Path) -> pd.DataFrame:
    rows = []

    for ds_dir in _result_dirs(results_dir):
        with open(ds_dir / "metrics.json") as fh:
            rows.append(json.load(fh))

    if not rows:
        return pd.DataFrame()

    return pd.DataFrame(rows)


def plot_auc_summary(metrics_df: pd.DataFrame, output_dir: Path) -> None:
    plot_df = metrics_df.sort_values("AUC", ascending=False).copy()

    fig, ax = plt.subplots(figsize=(12, 5))
    sns.barplot(data=plot_df, x="dataset", y="AUC", ax=ax)
    ax.set_title("KitNET replication: AUROC by dataset")
    ax.set_xlabel("Dataset")
    ax.set_ylabel("AUROC")
    ax.tick_params(axis="x", rotation=35)
    ax.axhline(0.5, color="red", linestyle="--", linewidth=1.0, alpha=0.7)

    fig.tight_layout()
    fig.savefig(output_dir / "summary_auc.png", dpi=180)
    plt.close(fig)


def plot_auprc_summary(metrics_df: pd.DataFrame, output_dir: Path) -> None:
    plot_df = metrics_df.sort_values("AUPRC", ascending=False).copy()

    fig, ax = plt.subplots(figsize=(12, 5))
    sns.barplot(data=plot_df, x="dataset", y="AUPRC", ax=ax)
    ax.set_title("KitNET replication: AUPRC by dataset")
    ax.set_xlabel("Dataset")
    ax.set_ylabel("AUPRC")
    ax.tick_params(axis="x", rotation=35)

    fig.tight_layout()
    fig.savefig(output_dir / "summary_auprc.png", dpi=180)
    plt.close(fig)


def plot_runtime_summary(metrics_df: pd.DataFrame, output_dir: Path) -> None:
    if "runtime_sec" not in metrics_df.columns:
        return

    plot_df = metrics_df.sort_values("runtime_sec", ascending=False).copy()

    fig, ax = plt.subplots(figsize=(12, 5))
    sns.barplot(data=plot_df, x="dataset", y="runtime_sec", ax=ax)
    ax.set_title("KitNET replication: runtime by dataset")
    ax.set_xlabel("Dataset")
    ax.set_ylabel("Seconds")
    ax.tick_params(axis="x", rotation=35)

    fig.tight_layout()
    fig.savefig(output_dir / "summary_runtime.png", dpi=180)
    plt.close(fig)


def plot_combined_roc(results_dir: Path, output_dir: Path) -> None:
    fig, ax = plt.subplots(figsize=(8, 8))

    for ds_dir in _result_dirs(results_dir):
        roc_path = ds_dir / "roc_curve.csv"
        if not roc_path.exists():
            continue

        roc_df = pd.read_csv(roc_path)
        with open(ds_dir / "metrics.json") as fh:
            metrics = json.load(fh)

        auc = metrics.get("AUC", np.nan)
        ax.plot(roc_df["fpr"], roc_df["tpr"], label=f"{ds_dir.name} (AUC={auc:.3f})")

    ax.plot([0, 1], [0, 1], linestyle="--", color="gray", linewidth=1.0)
    ax.set_title("KitNET replication: ROC curves")
    ax.set_xlabel("False positive rate")
    ax.set_ylabel("True positive rate")
    ax.legend(fontsize=8, loc="lower right")

    fig.tight_layout()
    fig.savefig(output_dir / "combined_roc.png", dpi=180)
    plt.close(fig)


def plot_combined_pr(results_dir: Path, output_dir: Path) -> None:
    fig, ax = plt.subplots(figsize=(8, 8))

    for ds_dir in _result_dirs(results_dir):
        pr_path = ds_dir / "pr_curve.csv"
        if not pr_path.exists():
            continue

        pr_df = pd.read_csv(pr_path)
        with open(ds_dir / "metrics.json") as fh:
            metrics = json.load(fh)

        auprc = metrics.get("AUPRC", np.nan)
        ax.plot(pr_df["recall"], pr_df["precision"], label=f"{ds_dir.name} (AP={auprc:.3f})")

    ax.set_title("KitNET replication: PR curves")
    ax.set_xlabel("Recall")
    ax.set_ylabel("Precision")
    ax.legend(fontsize=8, loc="lower left")

    fig.tight_layout()
    fig.savefig(output_dir / "combined_pr.png", dpi=180)
    plt.close(fig)


def plot_score_distributions(results_dir: Path, output_dir: Path) -> None:
    for ds_dir in _result_dirs(results_dir):
        score_df = pd.read_csv(ds_dir / "scores.csv")

        if score_df["label"].nunique() < 2:
            continue

        fig, ax = plt.subplots(figsize=(8, 5))
        sns.histplot(
            data=score_df,
            x="score",
            hue="label",
            bins=100,
            stat="density",
            common_norm=False,
            element="step",
            ax=ax,
        )
        ax.set_title(f"{ds_dir.name}: score distribution")
        ax.set_xlabel("Anomaly score")
        ax.set_ylabel("Density")
        ax.legend(["Benign", "Attack"])

        fig.tight_layout()
        fig.savefig(output_dir / f"{ds_dir.name}_score_dist.png", dpi=180)
        plt.close(fig)


def plot_score_timelines(results_dir: Path, output_dir: Path, max_points: int = 50000) -> None:
    for ds_dir in _result_dirs(results_dir):
        score_df = pd.read_csv(ds_dir / "scores.csv")

        if score_df.empty:
            continue

        step = max(1, len(score_df) // max_points)
        plot_df = score_df.iloc[::step].copy()

        fig, ax = plt.subplots(figsize=(14, 4))

        benign = plot_df[plot_df["label"] == 0]
        attack = plot_df[plot_df["label"] == 1]

        ax.scatter(benign["row_index"], benign["score"], s=4, alpha=0.35, label="Benign")
        ax.scatter(attack["row_index"], attack["score"], s=6, alpha=0.50, label="Attack")

        ax.set_title(f"{ds_dir.name}: anomaly score over stream")
        ax.set_xlabel("Row index")
        ax.set_ylabel("Score")
        ax.legend()

        fig.tight_layout()
        fig.savefig(output_dir / f"{ds_dir.name}_timeline.png", dpi=180)
        plt.close(fig)


def make_all_plots(results_dir: str | Path, output_dir: str | Path) -> None:
    results_dir = Path(results_dir)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    metrics_df = load_summary_metrics(results_dir)
    if metrics_df.empty:
        logger.warning("I could not find any finished result folders in %s", results_dir)
        return

    plot_auc_summary(metrics_df, output_dir)
    plot_auprc_summary(metrics_df, output_dir)
    plot_runtime_summary(metrics_df, output_dir)
    plot_combined_roc(results_dir, output_dir)
    plot_combined_pr(results_dir, output_dir)
    plot_score_distributions(results_dir, output_dir)
    plot_score_timelines(results_dir, output_dir)

    metrics_df.sort_values("AUC", ascending=False).to_csv(
        output_dir / "summary_metrics_sorted.csv",
        index=False,
    )


def main(argv: list[str] | None = None) -> None:
    parser = argparse.ArgumentParser(description="Plot KitNET replication results.")
    parser.add_argument("--results_dir", type=Path, required=True)
    parser.add_argument("--output_dir", type=Path, required=True)
    args = parser.parse_args(argv)

    make_all_plots(args.results_dir, args.output_dir)


if __name__ == "__main__":
    main()


In [13]:
%%writefile /kaggle/working/project/dataset_reader.py
from __future__ import annotations

import csv
import logging
from dataclasses import dataclass
from pathlib import Path

import numpy as np

log = logging.getLogger(__name__)

EXPECTED_N_FEATURES = 115

# String tokens that unambiguously indicate a benign sample
_BENIGN_VOCAB = frozenset({
    "0", "0.0", "benign", "normal", "false", "no", "clean", "legit",
})


@dataclass(frozen=True)
class DatasetPair:
    name:          str
    features_path: Path
    labels_path:   Path


def discover_dataset_pairs(search_dir: str | Path) -> list[DatasetPair]:
    # Finds all *_dataset.csv / *_labels.csv pairs in a directory.
    search_dir = Path(search_dir)
    if not search_dir.exists():
        raise FileNotFoundError(f"Directory not found: {search_dir}")

    found: list[DatasetPair] = []
    for feat_file in sorted(search_dir.glob("*_dataset.csv")):
        label_file = feat_file.with_name(
            feat_file.name.replace("_dataset.csv", "_labels.csv")
        )
        if not label_file.exists():
            log.warning("Skipping '%s': no matching label file.", feat_file.name)
            continue
        stem = feat_file.stem.replace("_dataset", "")
        found.append(DatasetPair(name=stem, features_path=feat_file, labels_path=label_file))
    return found


class PairedCSVDatasetReader:
    # Streams paired (feature_vector, label) rows from two aligned CSV files.

    def __init__(
        self,
        features_path:       str | Path,
        labels_path:         str | Path,
        expected_n_features: int = EXPECTED_N_FEATURES,
    ):
        self.features_path       = Path(features_path)
        self.labels_path         = Path(labels_path)
        self.expected_n_features = expected_n_features
        self._detected_width:    int | None = None

        if not self.features_path.exists():
            raise FileNotFoundError(f"Features file not found: {self.features_path}")
        if not self.labels_path.exists():
            raise FileNotFoundError(f"Labels file not found: {self.labels_path}")

    def __iter__(self):
        with (
            open(self.features_path, newline="") as feat_fh,
            open(self.labels_path,   newline="") as lbl_fh,
        ):
            feat_reader = csv.reader(feat_fh)
            lbl_reader  = csv.reader(lbl_fh)
            row_num     = 0

            while True:
                feat_row = self._advance(feat_reader)
                lbl_row  = self._advance(lbl_reader)

                if feat_row is None and lbl_row is None:
                    break

                if feat_row is None or lbl_row is None:
                    raise ValueError(
                        f"Row count mismatch near row {row_num} in '{self.features_path.name}'."
                    )

                yield row_num, self._to_features(feat_row, row_num), self._to_label(lbl_row, row_num)
                row_num += 1

    def count_rows(self) -> int:
        total = 0
        with open(self.features_path, newline="") as fh:
            for row in csv.reader(fh):
                if row:
                    total += 1
        return total

    @staticmethod
    def _advance(reader) -> list[str] | None:
        for row in reader:
            if row:
                return row
        return None

    def _to_features(self, row: list[str], row_num: int) -> np.ndarray:
        try:
            vec = np.asarray(row, dtype=np.float32)
        except ValueError as exc:
            raise ValueError(
                f"Non-numeric value in feature row {row_num} of '{self.features_path.name}'."
            ) from exc

        if self._detected_width is None:
            self._detected_width = len(vec)
            if self._detected_width != self.expected_n_features:
                log.warning(
                    "'%s' has %d features; expected %d.",
                    self.features_path.name, self._detected_width, self.expected_n_features,
                )

        return np.nan_to_num(vec, nan=0.0, posinf=0.0, neginf=0.0)

    def _to_label(self, row: list[str], row_num: int) -> int:
        raw = str(row[0]).strip()
        if not raw:
            raise ValueError(f"Empty label at row {row_num} in '{self.labels_path.name}'.")

        try:
            return int(float(raw) > 0.0)
        except ValueError:
            return 0 if raw.lower() in _BENIGN_VOCAB else 1


Overwriting /kaggle/working/project/dataset_reader.py


In [12]:
%%writefile /kaggle/working/project/run_replication.py
from __future__ import annotations

import argparse
import json
import logging
import sys
import time
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd

_root = str(Path(__file__).resolve().parent)
if _root not in sys.path:
    sys.path.insert(0, _root)

from core.feature_mapper import FeatureMapper
from core.kitnet import KitNET
from dataset_reader import (
    EXPECTED_N_FEATURES,
    DatasetPair,
    PairedCSVDatasetReader,
    discover_dataset_pairs,
)
from evaluation.metrics import build_curve_frames, compute_metrics, print_metrics
from evaluation.plot_results import make_all_plots

logging.basicConfig(
    format="%(asctime)s | %(levelname)-7s | %(message)s",
    datefmt="%H:%M:%S",
    level=logging.INFO,
)
_log = logging.getLogger(__name__)


class AnomalyDetectionPipeline:
    # Three-phase pipeline: FeatureMapper training → KitNET training → inference.
    # No score is returned during the two warm-up phases.

    def __init__(
        self,
        fm_grace:         int,
        ad_grace:         int,
        max_cluster_size: int   = 10,
        num_features:     int   = EXPECTED_N_FEATURES,
        beta_ratio:       float = 0.75,
        step_size:        float = 0.1,
    ) -> None:
        self.fm_grace  = fm_grace
        self.ad_grace  = ad_grace

        self.feat_mapper = FeatureMapper(
            n_features=num_features,
            max_cluster_size=max_cluster_size,
        )
        self.detector: Optional[KitNET] = None
        self._mapper_ready    = False
        self._rows_seen       = 0
        self.peak_train_score: float = 0.0
        self._beta = beta_ratio
        self._lr   = step_size

    @property
    def warmup_length(self) -> int:
        return self.fm_grace + self.ad_grace

    def step(self, feature_vec: np.ndarray) -> Tuple[Optional[float], str]:
        idx = self._rows_seen
        self._rows_seen += 1

        # Phase 1 — build feature correlation matrix
        if idx < self.fm_grace:
            self.feat_mapper.update(feature_vec)
            return None, "fm_train"

        # Transition — fit mapper once, then initialise KitNET
        if not self._mapper_ready:
            _log.info("Fitting FeatureMapper on %d rows...", self.fm_grace)
            self.feat_mapper.fit()
            self.detector = KitNET(
                cluster_sizes=self.feat_mapper.cluster_sizes,
                beta=self._beta,
                lr=self._lr,
            )
            self._mapper_ready = True
            _log.info(
                "FeatureMapper ready — k=%d clusters: %s",
                self.feat_mapper.n_clusters,
                self.feat_mapper.cluster_sizes,
            )

        sub_vecs = self.feat_mapper.transform(feature_vec)

        # Phase 2 — train KitNET autoencoders
        if idx < self.warmup_length:
            score = float(self.detector.train(sub_vecs))
            self.peak_train_score = max(self.peak_train_score, score)
            return None, "ad_train"

        # Phase 3 — inference
        return float(self.detector.execute(sub_vecs)), "exec"


def _safe_dirname(name: str) -> str:
    return name.replace(" ", "_")


def evaluate_single_dataset(
    pair:          DatasetPair,
    output_root:   Path,
    fm_grace:      int,
    ad_grace:      int,
    cluster_limit: int,
    beta:          float,
    learn_rate:    float,
    n_features:    int,
    log_every:     int = 100_000,
) -> Dict:
    _log.info("*" * 70)
    _log.info("Dataset  : %s", pair.name)
    _log.info("Features : %s", pair.features_path.name)
    _log.info("Labels   : %s", pair.labels_path.name)

    reader     = PairedCSVDatasetReader(pair.features_path, pair.labels_path, n_features)
    total_rows = reader.count_rows()

    pipeline = AnomalyDetectionPipeline(
        fm_grace=fm_grace, ad_grace=ad_grace,
        max_cluster_size=cluster_limit,
        num_features=n_features, beta_ratio=beta, step_size=learn_rate,
    )

    phase_counts: Dict[str, int] = {"fm_train": 0, "ad_train": 0, "exec": 0}
    scored_rows:  List[Dict]     = []
    t_start = time.time()

    for idx, feat_vec, true_label in reader:
        score, phase = pipeline.step(feat_vec)
        phase_counts[phase] += 1

        if phase == "exec":
            scored_rows.append({"row_index": idx, "label": int(true_label), "score": float(score)})

        if idx > 0 and idx % log_every == 0:
            _log.info(
                "Row %d / %d  |  fm=%d  ad=%d  exec=%d",
                idx, total_rows,
                phase_counts["fm_train"], phase_counts["ad_train"], phase_counts["exec"],
            )

    elapsed = time.time() - t_start

    if not scored_rows:
        _log.warning("No scored rows for '%s'.", pair.name)
        return {"dataset": pair.name, "error": "no_eval_rows"}

    results_df = pd.DataFrame(scored_rows)
    score_arr  = results_df["score"].to_numpy(dtype=np.float64)
    label_arr  = results_df["label"].to_numpy(dtype=np.int32)

    metrics = compute_metrics(
        scores=score_arr, labels=label_arr,
        dataset_name=pair.name, runtime_sec=elapsed,
        extra={
            "total_rows":       int(total_rows),
            "fm_grace":         int(fm_grace),
            "ad_grace":         int(ad_grace),
            "warmup_rows":      int(pipeline.warmup_length),
            "max_cluster_size": int(cluster_limit),
            "n_clusters":       int(pipeline.feat_mapper.n_clusters),
            "cluster_sizes":    pipeline.feat_mapper.cluster_sizes,
            "fm_train_rows":    int(phase_counts["fm_train"]),
            "ad_train_rows":    int(phase_counts["ad_train"]),
            "exec_rows":        int(phase_counts["exec"]),
            "peak_train_score": round(float(pipeline.peak_train_score), 6),
        },
    )

    print_metrics(metrics)

    target_dir = output_root / _safe_dirname(pair.name)
    target_dir.mkdir(parents=True, exist_ok=True)

    results_df.to_csv(target_dir / "scores.csv", index=False)
    np.save(target_dir / "scores.npy", score_arr)
    np.save(target_dir / "labels.npy", label_arr)

    roc_df, pr_df = build_curve_frames(scores=score_arr, labels=label_arr)
    roc_df.to_csv(target_dir / "roc_curve.csv", index=False)
    pr_df.to_csv( target_dir / "pr_curve.csv",  index=False)

    with open(target_dir / "metrics.json", "w") as fh:
        json.dump(metrics, fh, indent=2)

    _log.info("Artefacts written → %s", target_dir)
    return metrics


def resolve_datasets(
    scan_dir:      Optional[Path],
    features_file: Optional[Path],
    labels_file:   Optional[Path],
) -> List[DatasetPair]:
    if scan_dir is not None:
        pairs = discover_dataset_pairs(scan_dir)
        if not pairs:
            raise FileNotFoundError(f"No dataset pairs found in '{scan_dir}'.")
        return pairs

    if not features_file or not labels_file:
        raise ValueError("Provide both --features and --labels when not using --sample_dir.")

    stem = features_file.stem
    if stem.endswith("_dataset"):
        stem = stem[:-8]
    return [DatasetPair(name=stem, features_path=features_file, labels_path=labels_file)]


def main(argv: Optional[List[str]] = None) -> None:
    parser = argparse.ArgumentParser(
        description="Run KitNET anomaly detection on pre-extracted CSV datasets."
    )

    src = parser.add_mutually_exclusive_group(required=True)
    src.add_argument("--sample_dir", type=Path,
                     help="Directory with *_dataset.csv / *_labels.csv pairs.")
    src.add_argument("--features",   type=Path, help="Path to a features CSV.")

    parser.add_argument("--labels",      type=Path,  default=None)
    parser.add_argument("--output_dir",  type=Path,  default=Path("results"))
    parser.add_argument("--fm_grace",    type=int,   default=5_000)
    parser.add_argument("--ad_grace",    type=int,   default=50_000)
    parser.add_argument("--m",           type=int,   default=10)
    parser.add_argument("--beta",        type=float, default=0.75)
    parser.add_argument("--lr",          type=float, default=0.1)
    parser.add_argument("--expected_n_features", type=int, default=EXPECTED_N_FEATURES)
    parser.add_argument("--skip_plots",  action="store_true")

    args = parser.parse_args(argv)

    if args.features is not None and args.labels is None:
        parser.error("--labels is required when --features is provided.")

    args.output_dir.mkdir(parents=True, exist_ok=True)

    dataset_list = resolve_datasets(args.sample_dir, args.features, args.labels)
    _log.info("Datasets to process: %d", len(dataset_list))

    all_metrics: List[Dict] = []
    for ds in dataset_list:
        result = evaluate_single_dataset(
            pair=ds, output_root=args.output_dir,
            fm_grace=args.fm_grace, ad_grace=args.ad_grace,
            cluster_limit=args.m, beta=args.beta,
            learn_rate=args.lr, n_features=args.expected_n_features,
        )
        all_metrics.append(result)

    summary_df = pd.DataFrame(all_metrics)
    summary_df.to_csv(args.output_dir / "summary_metrics.csv", index=False)
    with open(args.output_dir / "summary_metrics.json", "w") as fh:
        json.dump(all_metrics, fh, indent=2)

    _log.info("Summary → %s", args.output_dir / "summary_metrics.json")

    if not args.skip_plots:
        plots_dir = args.output_dir / "_plots"
        make_all_plots(args.output_dir, plots_dir)
        _log.info("Plots → %s", plots_dir)


if __name__ == "__main__":
    main()


Overwriting /kaggle/working/project/run_replication.py


In [ ]:
!python /kaggle/working/project/run_replication.py \
  --sample_dir /kaggle/working/kitsune_20pct_samples \
  --output_dir /kaggle/working/project/results \
  --fm_grace 5000 \
  --ad_grace 50000 \
  --m 10

In [11]:
import shutil

shutil.make_archive(
    "/kaggle/working/results_v2",   # output name
    "zip",                                # format
    "/kaggle/working/project/results"    # target folder
)


'/kaggle/working/results_v2.zip'

In [4]:
!python /kaggle/working/project/run_replication.py \
    --features /kaggle/working/kitsune_20pct_samples/Mirai_Botnet_20pct_dataset.csv \
    --labels   /kaggle/working/kitsune_20pct_samples/Mirai_Botnet_20pct_labels.csv \
    --output_dir /kaggle/working/project/results \
    --fm_grace 2000 \
    --ad_grace 18000 \
    --m 10 \
    --skip_plots


INFO I found 1 dataset(s) to run.
INFO ======================================================================
INFO Dataset: Mirai_Botnet_20pct
INFO Features: Mirai_Botnet_20pct_dataset.csv
INFO Labels:   Mirai_Botnet_20pct_labels.csv
INFO Fitting the FeatureMapper after 2000 rows.
INFO FeatureMapper ready: k=12, cluster sizes=[10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 5]
INFO Row 100000/152339 | fm=2000 ad=18000 exec=80001
INFO ────────────────────────────────────────────────────────────
INFO Dataset: Mirai_Botnet_20pct
INFO Eval rows: 132339 | Benign: 4234 | Attack: 128105
INFO AUC: 0.9412 | AUPRC: 0.9980 | EER: 0.1094
INFO TPR@FPR=0: 0.8814 | TPR@FPR=0.001: 0.8815
INFO F1: 0.9837 | Precision: 0.9680 | Recall: 1.0000
INFO TP=128105 FP=4234 FN=0 TN=0
INFO Runtime: 134.31s | Rows/sec: 985.36
INFO ────────────────────────────────────────────────────────────
INFO Saved dataset results to /kaggle/working/project/results/Mirai_Botnet_20pct
INFO Saved the summary table to /kaggle/working/p

In [5]:
import json
import pandas as pd
from pathlib import Path

results_dir = Path("/kaggle/working/project/results")

rows = []
for metrics_file in sorted(results_dir.glob("*/metrics.json")):
    with open(metrics_file) as f:
        rows.append(json.load(f))

summary_df = pd.DataFrame(rows)
summary_df.to_csv(results_dir / "summary_metrics.csv", index=False)

with open(results_dir / "summary_metrics.json", "w") as f:
    json.dump(rows, f, indent=2)

print(f"Summary rebuilt from {len(rows)} datasets:")
print(summary_df[["dataset", "AUC", "AUPRC", "F1_optimal"]].sort_values("AUC", ascending=False).to_string(index=False))


Summary rebuilt from 9 datasets:
                dataset      AUC    AUPRC  F1_optimal
       SSDP_Flood_20pct 0.999879 0.999620    0.999077
SSL_Renegotiation_20pct 0.984619 0.651036    0.747884
          OS_Scan_20pct 0.976124 0.507661    0.667771
          SYN_DoS_20pct 0.959001 0.031879    0.096055
     Mirai_Botnet_20pct 0.941199 0.998019    0.983743
  Video_Injection_20pct 0.924218 0.404739    0.575591
         ARP_MitM_20pct 0.716540 0.641364    0.748022
   Active_Wiretap_20pct 0.428164 0.404969    0.664846
          Fuzzing_20pct 0.361029 0.172866    0.417762


In [10]:
!python /kaggle/working/project/evaluation/plot_results.py \
    --results_dir /kaggle/working/project/results \
    --output_dir  /kaggle/working/project/results/_plots